# C–D · Path search và perturbation trên QA scorer thật

**Câu hỏi:** xóa đường có làm đổi điểm/hạng của ảnh đích trên cùng top-100 không? Chạy Colab GPU. `label_h` chỉ nhận `edge_index`, do đó kết quả chỉ chứng minh **relational connectivity**, không chứng minh mô hình hiểu tên predicate. Đường 2–3 cạnh đi có hướng từ concept query qua nhãn ảnh rồi `contains`; đường một cạnh `contains` là object evidence. Mọi case, kể cả không có đường, ở trong mẫu số coverage.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, importlib.util, json, subprocess, sys
import numpy as np
if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive')
else:
    ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'graphml-project3').is_dir())
REPO=ROOT/'graphml-project3'; DUONG=REPO/'duong'; RUNS=DUONG/'runs/qa_vg_psg_xai'
sys.path[:0]=[str(DUONG),str(REPO/'huy/src')]
if importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO/'huy')],check=True)
import torch
from xai.model import HuyQAPathCaseModel, MODEL_SPECS
from xai.path_protocol import evaluate_case
from xai.query_registry import validated_registry_rows
from xai.perturbation import rank_of
from mmxai.data.task1 import load_task1
from mmxai.graph.build import build_graph
from mmxai.graph.qa import QAData, QAScorer, qa_scores
from mmxai.experiments.t1_graph import load_eval_module
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)


## 1 · Chốt protocol và provenance

Mặc định chạy lại đúng 24 caption baseline. Chỉ chọn `verified40` sau khi ảnh dương/âm và nhóm query đã duyệt. Hai mức screen là phân tích độ nhạy được đặt trước. Budget là **số cạnh duy nhất**, random path khớp loại cạnh/độ dài/budget; thiếu đối chứng ghi N/A. Hash nguồn huấn luyện, bundle và checkpoint được xác minh; hash XAI mới khác baseline là có chủ đích.

In [ ]:
QUERY_MODE='baseline24'  # 'verified40' only after visual review
MODEL_SEEDS=(0,1,2); PATH_SCREEN_LIMITS=(40,120); EDGE_BUDGETS=(2,3,5)
RANDOM_REPEATS=20; RANDOM_SEED=20261121
MANIFEST=RUNS/'dual-20261121-125423_manifest.json'
manifest=json.loads(MANIFEST.read_text())
records={(r['dataset'],int(r['seed'])):r for r in manifest['runs']}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''): h.update(block)
    return h.hexdigest()
for rel,digest in manifest['source_sha256'].items():
    if rel.startswith('huy/'): assert sha256(REPO/rel)==digest,rel
for dataset,spec in MODEL_SPECS.items():
    bundle_dir=ROOT/spec.bundle_dir
    for rel,digest in manifest['bundle_sha256'][dataset].items():
        assert sha256(bundle_dir/rel)==digest,(dataset,rel)
    for seed in MODEL_SEEDS:
        rec=records[dataset,seed]
        ckpt=RUNS/spec.experiment/Path(rec['run_dir']).name/'model.pt'
        assert sha256(ckpt)==rec['checkpoint_sha256']
if QUERY_MODE=='baseline24':
    selected=json.loads((RUNS/'xai-20261121-130842/selected_cases.json').read_text())
else:
    selected=validated_registry_rows(DUONG/'review/queries_relational_verified.csv')
assert len(selected)==(24 if QUERY_MODE=='baseline24' else 40)
for dataset,spec in MODEL_SPECS.items():
    seed0_dir=RUNS/spec.experiment/Path(records[dataset,0]['run_dir']).name
    ranks0=json.loads((seed0_dir/'ranks_test_full.json').read_text())['ranks']
    subset=[r for r in selected if r['dataset']==dataset]
    success=sum(int(ranks0[int(r['candidate_row'])])==1 for r in subset)
    failure=sum(int(ranks0[int(r['candidate_row'])])>1 for r in subset)
    assert success>=5 and failure>=5,(dataset,success,failure)
    print(dataset,'seed-0 success/failure:',success,failure)
OUT=RUNS/('path-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'protocol.json').write_text(json.dumps({'query_mode':QUERY_MODE,'manifest':str(MANIFEST),'seeds':MODEL_SEEDS,'path_screen_limits':PATH_SCREEN_LIMITS,'edge_budgets':EDGE_BUDGETS,'random_repeats':RANDOM_REPEATS,'random_seed':RANDOM_SEED,'selected_query_ids':{d:[int(r['caption_id'] if QUERY_MODE=='baseline24' else r['query_id']) for r in selected if r['dataset']==d] for d in MODEL_SPECS}},indent=2))
print('Output:',OUT)


## 2 · Score gốc, vùng đủ điều kiện và candidate path

`E_eligible` gồm mọi `contains` của ảnh đích và các cạnh nhãn–nhãn trong incoming receptive field thật (số tầng GNN). Screen chỉ dùng topology và giới hạn path đã khóa; cạnh ngoài screen vẫn hiện diện trong scorer. Với từng seed/query/screen, kiểm tra QA score và hạng GT so với checkpoint.

In [ ]:
result_rows=[]; path_rows=[]; case_rows=[]
for dataset,spec in MODEL_SPECS.items():
    bundle_dir=ROOT/spec.bundle_dir
    ev=load_eval_module(bundle_dir); protocol=ev.Protocol.load(bundle_dir)
    bundle=load_task1(bundle_dir,bundle_dir/'huy_extra_cn')
    cap_concepts={int(c['caption_id']):c['concepts'] for c in bundle.captions}
    cands=protocol.candidates('test',None)
    chosen=[r for r in selected if r['dataset']==dataset]
    image_tensor=torch.from_numpy(bundle.image_emb).to(DEVICE)
    for seed in MODEL_SEEDS:
        rec=records[dataset,seed]
        run_dir=RUNS/spec.experiment/Path(rec['run_dir']).name
        cfg=json.loads((run_dir/'config.json').read_text())
        graph=build_graph(bundle,variant=cfg['graph']['variant'],min_rel_count=int(cfg['graph'].get('min_rel_count',1)),seed=seed,rel_merge=cfg['graph'].get('rel_merge','directed'))
        assert cfg['graph']['variant']==spec.graph_variant and cfg['gnn']['kind']==spec.gnn_kind
        assert graph.stats['n_label']==spec.expected_labels and graph.stats['E_concept']==spec.expected_concept_edges
        qd=QAData(graph,cap_concepts,DEVICE)
        model=QAScorer(bundle.image_emb.shape[1],**cfg['gnn']).to(DEVICE)
        model.load_state_dict(torch.load(run_dir/'model.pt',map_location='cpu',weights_only=True)); model.eval()
        saved_ranks=json.loads((run_dir/'ranks_test_full.json').read_text())['ranks']
        for item in chosen:
            j=int(item['candidate_row']); cap_id=int(item['caption_id'] if QUERY_MODE=='baseline24' else item['query_id'])
            assert int(cands['q_cap'][j])==cap_id
            query=protocol.captions[cap_id]['text']
            assert QUERY_MODE=='baseline24' or query==item['query']
            gt_id=str(int(cands['q_img'][j]))
            group=str(cands['group'][j]) if QUERY_MODE=='baseline24' else item['verified_group']
            for screen_limit in PATH_SCREEN_LIMITS:
                case=HuyQAPathCaseModel(model,graph,qd,bundle,protocol,cands,ev,j,rec['alpha'],DEVICE,max_paths=screen_limit,image_tensor=image_tensor)
                reference=qa_scores(model,qd,image_tensor,np.asarray(protocol.cap[cap_id:cap_id+1]),np.asarray([cap_id]),case.rows_np[None],DEVICE)[0]
                np.testing.assert_allclose(case.score_removed()['qa'],reference,rtol=1e-4,atol=1e-5)
                ids=tuple(map(str,case.ids)); before=case.scorer.score(query,ids,case.intervention_graph)
                if gt_id in ids: assert rank_of(before,gt_id,ids)==int(saved_ranks[j])
                else: assert int(saved_ranks[j])>len(ids)
                rr,pp=evaluate_case(case,query,gt_id,dataset=dataset,model_seed=seed,query_id=str(cap_id),group=group,budgets=EDGE_BUDGETS,random_repeats=RANDOM_REPEATS,random_seed=RANDOM_SEED+seed*100000+j*10+screen_limit,hard_negative_image_id=(item['hard_negative_candidate_id'] if QUERY_MODE=='verified40' else None))
                for r in rr: r['screen_path_limit']=screen_limit
                for p in pp: p.update(dataset=dataset,model_seed=seed,screen_path_limit=screen_limit,query=query,ground_truth_image_id=gt_id)
                result_rows.extend(rr); path_rows.extend(pp)
                def source_url(im): return im.get('url') or ('https://images.cocodataset.org/'+im['coco_file'] if im.get('coco_file') else im.get('coco_url',''))
                case_rows.append({'dataset':dataset,'query_id':cap_id,'query':query,'source_group':group,'model_seed':seed,'screen_path_limit':screen_limit,'gt_image_id':gt_id,'retrieved_image_id':case.top_id,'retrieval_outcome':'success' if case.top_id==int(gt_id) else 'failure','gt_rank':int(saved_ranks[j]),'target_rank':1,'gt_source_url':source_url(protocol.images[int(gt_id)]),'retrieved_source_url':source_url(protocol.images[case.top_id])})
        print('Completed:',dataset,seed,len(chosen),'queries')


## 3 · Lưu artifact bất biến

`paths.jsonl`: mọi path đã sàng lọc, contribution cả path, lý do chọn và `no_path`. `xai_results.csv`: mỗi can thiệp trên cùng candidate pool, score graph/QA/fuse, hạng ảnh đích và GT riêng, mẫu số eligible/screened theo loại. `case_manifest.csv` giữ URL ảnh cho workbook. Không ghi đè baseline.

In [ ]:
assert result_rows and path_rows and case_rows
with (OUT/'xai_results.csv').open('w',newline='',encoding='utf8') as f:
    keys=list(dict.fromkeys(k for row in result_rows for k in row)); w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(result_rows)
with (OUT/'case_manifest.csv').open('w',newline='',encoding='utf8') as f:
    w=csv.DictWriter(f,fieldnames=list(case_rows[0])); w.writeheader(); w.writerows(case_rows)
with (OUT/'paths.jsonl').open('w',encoding='utf8') as f:
    for row in path_rows: f.write(json.dumps(row,ensure_ascii=False)+'\n')
(OUT/'complete.json').write_text(json.dumps({'n_result_rows':len(result_rows),'n_path_rows':len(path_rows),'n_cases':len(case_rows)},indent=2))
print('Saved:',OUT,len(result_rows),len(path_rows),len(case_rows))
